# 第2日 ノートブック03：OpenCV と距離計測

# Day 2 Notebook 03: OpenCV and Distance Measurement

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. OpenCV で画像を読み込み・表示・保存できる | 1. Read, display and save images with OpenCV |
| 2. グレースケール変換と 2 値化の意味を説明できる | 2. Explain greyscale conversion and thresholding |
| 3. 輪郭（contour）を検出して位置を求められる | 3. Find contours and obtain their positions |
| 4. 2 点間の距離をピクセルで計算できる | 4. Compute the distance between two points in pixels |
| 5. ピクセルをミリメートルに変換し、精度の限界を説明できる | 5. Convert pixels to millimetres and explain the accuracy limits |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 120 分 / 120 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第2日ノートブック01・02 を終えていること。 / You have finished Day 2 Notebooks 01 and 02. |
| 必要なライブラリ / Required libraries | opencv-python, numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| エッジ・輪郭の検出結果 | Edge and contour results |
| ピクセル→mm の換算結果と、測定誤差の考察 | Pixel-to-mm results with a discussion of measurement error |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：画像処理の流れの説明 | Instructor: the image-processing pipeline |
| 09:00–10:30 | Part 1〜Part 3（読み込み・グレー・2 値化） | Part 1-Part 3 (read, grey, threshold) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 6（輪郭・距離・校正） | Part 4-Part 6 (contours, distance, calibration) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
    ("cv2", "opencv-python"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

#### 日本語：OpenCV の色の順番に注意
**日本語：** OpenCV は色を **BGR**（青・緑・赤）の順で扱います。
Matplotlib は **RGB**（赤・緑・青）です。

変換せずに表示すると、赤と青が入れ替わって見えます。**必ず変換してください。**

```python
image_bgr = cv2.imread(path)                       # BGR で読み込まれる
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)   # RGB に変換
```

**日本語：** 研修全体で、**表示は Matplotlib、処理は OpenCV** と決めます。
`cv2.imshow()` は Jupyter で固まることがあるので使いません。

**English:** OpenCV stores colours as **BGR** (blue, green, red); Matplotlib expects **RGB**.
Without conversion, red and blue appear swapped. Always convert:

```python
image_bgr = cv2.imread(path)                             # read as BGR
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)   # convert to RGB
```

Rules for this course: **OpenCV for processing, Matplotlib for display.** Never use `cv2.imshow()`
in a notebook.
#### English: Watch out for OpenCV's colour order

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("OpenCV version:", cv2.__version__)

def show(images, titles=None, figsize=(14, 4), cmap=None):
    """複数の画像を並べて表示する / show several images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 3:
            ax.imshow(image)
        else:
            ax.imshow(image, cmap=cmap or "gray", vmin=0, vmax=255)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

---

## Part 1　画像を読み込む・保存する / Reading and saving images

#### 日本語：3 つの基本操作
**日本語：**

| 操作 | 関数 | 戻り値 |
|---|---|---|
| 読み込む | `cv2.imread(path)` | NumPy 配列（BGR）。失敗すると `None` |
| 表示する | `plt.imshow(...)` | — |
| 保存する | `cv2.imwrite(path, image)` | 成功で `True` |

> **日本語：** `cv2.imread` は、**ファイルが無いとエラーではなく `None` を返します**。
> 必ず `None` かどうかを確認してください。ここは初心者が最も見落とす点です。

**English:**

| Operation | Function | Returns |
|---|---|---|
| read | `cv2.imread(path)` | a BGR NumPy array, or `None` on failure |
| display | `plt.imshow(...)` | — |
| save | `cv2.imwrite(path, image)` | `True` on success |

`cv2.imread` does **not** raise an error for a missing file; it returns `None`. Always check.
This is the most commonly missed detail for beginners.
#### English: The three basic operations

In [ ]:
image_path = ASSETS / "defect_samples" / "defect_multi.png"

image_bgr = cv2.imread(str(image_path))

if image_bgr is None:
    print("画像が読めませんでした / could not read:", image_path)
    print("assets\\scripts\\generate_training_assets.py を実行してください。")
else:
    print("読み込み成功 / loaded")
    print("shape (height, width, channels):", image_bgr.shape)
    print("dtype:", image_bgr.dtype)
    print("最初の画素の BGR 値 / first pixel BGR:", image_bgr[0, 0])

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    show([image_bgr, image_rgb],
         ["BGR as-is (wrong colours) / BGR のまま表示", "converted to RGB / RGB に変換"],
         figsize=(11, 4))

#### 日本語：保存する
**日本語：** `cv2.imwrite` で保存します。保存先のフォルダは **先に作る** 必要があります。

研修では、**元の素材を上書きしない** ように、結果は `submissions` の中に保存します。

**English:** `cv2.imwrite` saves an image. The target folder must **already exist**.

Never overwrite the original assets: this course writes results into `submissions`.
#### English: Saving

In [ ]:
output_dir = COURSE_ROOT / "submissions" / "day02"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "opencv_rgb_copy.png"

if image_bgr is not None:
    saved = cv2.imwrite(str(output_path), image_bgr)
    print("保存できたか / saved:", saved)
    print("保存先 / path:", output_path)
    print("ファイルサイズ / size:", output_path.stat().st_size, "bytes")

---

## Part 2　グレースケールと 2 値化 / Greyscale and thresholding

#### 日本語：グレースケールにする
**日本語：** 色の情報は、欠陥の検出には多くの場合 **不要** です。
明るさだけにすると、計算が速く、扱いも簡単になります。

```python
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
```

`gray` は 2 次元（高さ, 幅）の配列になります。

**English:** Colour information is often **unnecessary** for defect detection. Using brightness
only is faster and simpler.

```python
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
```

`gray` is a two-dimensional array (height, width).
#### English: Converting to greyscale

#### 日本語：2 値化（しきい値処理）
**日本語：** 2 値化は、**しきい値より暗い画素を 0、明るい画素を 255** に変える処理です。
欠陥（暗い部分）だけを取り出せます。

```python
_, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
#                        ↑元画像 ↑しきい値 ↑最大値 ↑「反転」2値化
```

`THRESH_BINARY_INV` は **反転** です。次のようになります。

| 元の明るさ | `THRESH_BINARY` | `THRESH_BINARY_INV` |
|---|---|---|
| しきい値より明るい | 255（白） | 0（黒） |
| しきい値より暗い | 0（黒） | 255（白） |

**日本語：** 欠陥を **白** で表したいので、`THRESH_BINARY_INV` を使います。
こうすると、後の「輪郭検出」が「白い部分を探す」という素直な形になります。

**English:** Thresholding turns pixels darker than a threshold into 0 and brighter ones into 255,
so dark defects can be isolated.

```python
_, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
#                        ^input  ^thresh ^maxval ^inverted binary
```

| Original brightness | `THRESH_BINARY` | `THRESH_BINARY_INV` |
|---|---|---|
| brighter than threshold | 255 (white) | 0 (black) |
| darker than threshold | 0 (black) | 255 (white) |

We use `THRESH_BINARY_INV` so that **defects become white**, which makes the later contour search
simply "find the white regions".
#### English: Thresholding

In [ ]:
if image_bgr is not None:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    threshold_value = 150
    _, binary = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY_INV)

    print("gray shape   :", gray.shape)
    print("gray min/max :", int(gray.min()), "/", int(gray.max()))
    print("white pixels / 白い画素の数:", int((binary == 255).sum()))

    show([gray, binary], ["greyscale / グレースケール", f"binary (thresh={threshold_value}) / 2値化"],
         figsize=(11, 4))

#### 日本語：しきい値を変えると何が変わるか
**日本語：** しきい値は **人間が決める数値** です。ここが画像処理の難しさです。

- しきい値を **高くする**（例 200）→ 明るいムラまで欠陥と判定 → **誤検出が増える**
- しきい値を **低くする**（例 60）→ 薄い欠陥が残らない → **見逃しが増える**

**日本語：** 次に、いくつかの値で白い画素の数を数えてみましょう。

**English:** The threshold is a **number chosen by a human**, and that is the hard part.

- **raising** it (e.g. 200) flags bright mottling as a defect -> more **false alarms**
- **lowering** it (e.g. 60) drops faint defects -> more **misses**

Let us count white pixels for several thresholds.
#### English: What changes with the threshold

In [ ]:
if image_bgr is not None:
    print(f"{'しきい値 / threshold':>22}  {'白い画素 / white pixels':>24}")
    print("-" * 50)
    for value in [60, 100, 150, 180, 200, 220]:
        _, candidate = cv2.threshold(gray, value, 255, cv2.THRESH_BINARY_INV)
        print(f"{value:>22}  {int((candidate == 255).sum()):>24}")

---

## Part 3　輪郭の検出 / Finding contours

#### 日本語：輪郭とは、白い領域のかたまり
**日本語：** `cv2.findContours` は、2 値化画像の中の **白いかたまり** を 1 つずつ見つけます。

```python
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
```

| 引数 | 意味 |
|---|---|
| `cv2.RETR_EXTERNAL` | 一番外側の輪郭だけを返す |
| `cv2.CHAIN_APPROX_SIMPLE` | 輪郭の点を間引いて軽くする |

各輪郭から、次の情報が得られます。

| 関数 | 得られるもの |
|---|---|
| `cv2.contourArea(c)` | 面積（画素²） |
| `cv2.boundingRect(c)` | 外接する四角 (x, y, w, h) |
| `cv2.moments(c)` | 重心の計算に使う値 |

**English:** `cv2.findContours` finds each **connected white region** in a binary image.

```python
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
```

| Argument | Meaning |
|---|---|
| `cv2.RETR_EXTERNAL` | return only the outermost contours |
| `cv2.CHAIN_APPROX_SIMPLE` | compress the contour points |

From each contour you can obtain:

| Function | Gives |
|---|---|
| `cv2.contourArea(c)` | area in pixels² |
| `cv2.boundingRect(c)` | the enclosing rectangle (x, y, w, h) |
| `cv2.moments(c)` | values used to compute the centre |
#### English: A contour is one connected white region

In [ ]:
if image_bgr is not None:
    contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print("見つかった輪郭の数 / contour count:", len(contours))

    min_area = 5      # 小さすぎるノイズを無視する / ignore tiny noise
    kept = [c for c in contours if cv2.contourArea(c) >= min_area]
    print("面積で絞った後 / after the area filter:", len(kept))
    print()

    annotated = image_bgr.copy()
    for index, contour in enumerate(kept, start=1):
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        cv2.drawContours(annotated, [contour], -1, (0, 0, 255), 1)
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (255, 0, 0), 1)
        print(f"輪郭 {index}: 面積 / area = {area:7.1f} px2, bbox = (x={x}, y={y}, w={w}, h={h})")

    show([cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)],
         ["detected contours (red) and boxes (blue) / 検出した輪郭"],
         figsize=(7, 5))

---

## Part 4　2 点間の距離 / Distance between two points

#### 日本語：まず図で考えましょう
**日本語：** 2 点 A と B の距離を求めたいとします。
A から B へまっすぐ測るのは、画像の上では難しいですね。

そこで、**直角三角形** を作ります。

1. 横方向の差：`dx = x2 - x1`
2. 縦方向の差：`dy = y2 - y1`
3. 斜めの長さ（＝距離）は、三平方の定理で求まります

下の図を見てください。
#### English: Think with a picture first
**English:** Suppose you want the distance between points A and B. Measuring along the diagonal
directly is awkward on a grid.

So build a **right triangle**:

1. horizontal difference: `dx = x2 - x1`
2. vertical difference: `dy = y2 - y1`
3. the diagonal length (the distance) follows from Pythagoras

Look at the figure.

In [ ]:
from PIL import Image

diagram = Image.open(ASSETS / "diagrams" / "diagram_pixel_distance.png")
plt.figure(figsize=(8, 6))
plt.imshow(diagram)
plt.axis("off")
plt.show()

#### 日本語：式の意味
**日本語：** 式は次の 1 行です。

```text
距離 = √( dx² + dy² )
```

`dx²` は「dx を 2 乗する」、`√` は「平方根（ルート）」です。
Python では `** 0.5` または `math.sqrt` を使います。

```python
distance = (dx ** 2 + dy ** 2) ** 0.5
```

> **日本語：** 「2 乗して足して、ルートを取る」——これだけです。
> 公式を暗記する必要はありません。**直角三角形の斜辺** を求めているだけです。

**English:** The formula is one line:

```text
distance = sqrt( dx^2 + dy^2 )
```

`dx ** 2` squares `dx`, and `sqrt` takes the square root. In Python use `** 0.5` or `math.sqrt`.

```python
distance = (dx ** 2 + dy ** 2) ** 0.5
```

Square both sides, add, take the root. It is only the hypotenuse of a right triangle.
#### English: What the formula means

In [ ]:
def distance_px(point_a, point_b):
    """2 点間の距離をピクセルで返す / distance in pixels between two points.

    Parameters
    ----------
    point_a, point_b : tuple[int, int]
        (x, y) の画素座標 / pixel coordinates (x, y)
    """
    dx = point_b[0] - point_a[0]
    dy = point_b[1] - point_a[1]
    return (dx ** 2 + dy ** 2) ** 0.5


examples = [
    ((0, 0), (3, 4)),
    ((10, 10), (10, 10)),
    ((100, 50), (160, 130)),
    ((0, 0), (0, 25)),
]

for a, b in examples:
    d = distance_px(a, b)
    print(f"A={a}  B={b}  ->  distance = {d:.2f} px")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
A=(0, 0)  B=(3, 4)  ->  distance = 5.00 px
A=(10, 10)  B=(10, 10)  ->  distance = 0.00 px
A=(100, 50)  B=(160, 130)  ->  distance = 100.00 px
A=(0, 0)  B=(0, 25)  ->  distance = 25.00 px
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
A=(0, 0)  B=(3, 4)  ->  distance = 5.00 px
A=(10, 10)  B=(10, 10)  ->  distance = 0.00 px
A=(100, 50)  B=(160, 130)  ->  distance = 100.00 px
A=(0, 0)  B=(0, 25)  ->  distance = 25.00 px
```

#### 日本語：3-4-5 の確認
**日本語：** `(0,0)` と `(3,4)` の距離は **5** です。
`3² + 4² = 9 + 16 = 25`、`√25 = 5`。計算が合っていますね。

**English:** The distance between `(0,0)` and `(3,4)` is **5**, because `3² + 4² = 25` and
`sqrt(25) = 5`.
#### English: Verifying 3-4-5

---

## Part 5　ピクセルをミリメートルに変換する / Converting pixels to millimetres

#### 日本語：なぜ校正（キャリブレーション）が必要か
**日本語：** 画像の距離は **ピクセル** で測られます。
しかし現場で知りたいのは **ミリメートル** です。

同じ 100 mm の物体でも、カメラが遠ければ **小さく**、近ければ **大きく** 写ります。
つまり「1 ピクセルが何 mm か」は、**カメラと物体の位置関係で決まる** のです。

そこで、**長さが分かっている物**（基準スケール）を同じ画面に写して、
ピクセルと mm の比率を求めます。これを **校正** と呼びます。

```text
px_per_mm = 基準のピクセル長 ÷ 基準の実寸(mm)
```

**English:** Distances in an image are measured in **pixels**, but the shop floor needs
**millimetres**.

The same 100 mm object appears **smaller** when the camera is far away and **larger** when it is
close: "how many mm is one pixel" depends on the **camera-to-object geometry**.

So you image an object of known size (a reference scale) and derive the ratio. This is
**calibration**.

```text
px_per_mm = reference length in pixels / reference length in mm
```
#### English: Why calibration is necessary

In [ ]:
import json

calibration_path = ASSETS / "diagrams" / "calibration_image.png"
calibration_info_path = ASSETS / "diagrams" / "calibration_ground_truth.json"

calibration_image = cv2.imread(str(calibration_path))

if calibration_image is None:
    print("校正画像がありません / calibration image missing:", calibration_path)
    print("assets\\scripts\\generate_training_assets.py を実行してください。")
else:
    info = json.loads(calibration_info_path.read_text(encoding="utf-8"))
    reference_px = info["reference_px"]
    reference_mm = info["reference_mm"]

    px_per_mm = reference_px / reference_mm
    mm_per_px = reference_mm / reference_px

    print("基準の長さ（画素）/ reference pixels :", reference_px)
    print("基準の長さ（実寸）/ reference mm     :", reference_mm)
    print(f"px/mm = {reference_px} / {reference_mm} = {px_per_mm:.3f}")
    print(f"mm/px = {reference_mm} / {reference_px} = {mm_per_px:.5f}")

    show([cv2.cvtColor(calibration_image, cv2.COLOR_BGR2RGB)],
         ["calibration image with a 100 mm reference bar / 校正用画像"], figsize=(7, 5))

#### 日本語：校正の考え方の図
**日本語：** 下の図は、**同じ 100 mm の基準** が、カメラの距離によって
何ピクセルで写るかを示しています。

- 遠い：100 mm が 120 px → **1.20 px/mm**
- 近い：100 mm が 240 px → **2.40 px/mm**

**日本語：** つまり、校正値は **その撮影条件でしか使えません**。
カメラを動かしたら、**必ず校正し直す** 必要があります。

**English:** The figure shows the **same 100 mm reference** imaged at two camera distances:

- far: 100 mm occupies 120 px -> **1.20 px/mm**
- near: 100 mm occupies 240 px -> **2.40 px/mm**

A calibration value is therefore valid **only for that exact setup**. Move the camera and you must
calibrate again.
#### English: The idea of calibration

In [ ]:
concept = Image.open(ASSETS / "diagrams" / "diagram_calibration_concept.png")
plt.figure(figsize=(10, 4))
plt.imshow(concept)
plt.axis("off")
plt.show()

#### 日本語：測定してみる
**日本語：** 校正値を使って、2 点間の距離を mm で求めます。

```text
距離(mm) = 距離(px) × mm_per_px
```

下のセルでは、校正画像の基準線の両端を測り、**100 mm に戻るか** を確認します。
これが「校正が正しいか」の確認方法です。

**English:** Use the calibration to express a distance in mm:

```text
distance_mm = distance_px * mm_per_px
```

The next cell measures the two ends of the reference bar and checks that it comes back as
**100 mm**. This is how you verify your calibration.
#### English: Measure something

In [ ]:
if calibration_image is not None:
    left_point = (info["bar_x1"], info["bar_y"])
    right_point = (info["bar_x2"], info["bar_y"])

    measured_px = distance_px(left_point, right_point)
    measured_mm = measured_px * mm_per_px

    print("測った 2 点 / measured points:", left_point, right_point)
    print(f"距離 / distance : {measured_px:.2f} px")
    print(f"換算 / in mm    : {measured_mm:.3f} mm")
    print(f"真値 / truth    : {reference_mm} mm")
    print(f"誤差 / error    : {abs(measured_mm - reference_mm):.6f} mm")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
測った 2 点 / measured points: (100, 150) (220, 150)
距離 / distance : 120.00 px
換算 / in mm    : 100.000 mm
真値 / truth    : 100.0 mm
誤差 / error    : 0.000000 mm
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
measured points: (100, 150) (220, 150)
distance : 120.00 px
in mm    : 100.000 mm
truth    : 100.0 mm
error    : 0.000000 mm
```

#### 日本語：単一のスケール係数が使える条件
**日本語：** 「1 つの px/mm で全体を測る」のは、次の条件が満たされるときだけです。

| 条件 | 満たされないとどうなるか |
|---|---|
| カメラがシート面に **正対** している | 斜めだと、場所によって縮尺が変わる |
| レンズ歪みが小さい（または補正済み） | 画像の端ほど誤差が大きい |
| シートが平らで、カメラから **同じ距離** | 浮きや盛り上がりで縮尺が変わる |
| 視野全体で倍率が一定（望遠〜標準レンズ） | 広角レンズは周辺で縮尺が変わる |

**日本語：** つまり、**校正していない画像の測定値は、正しいとは言えません**。
「ピクセルで測った」だけでは、mm の保証はありません。

**English:** Using a single px/mm for the whole image is valid only when:

| Condition | If it fails |
|---|---|
| the camera faces the sheet **perpendicularly** | oblique views change the scale across the image |
| lens distortion is small or corrected | error grows towards the edges |
| the sheet is flat and at a **constant distance** | wrinkles or bulges change the scale |
| magnification is constant across the field of view | wide-angle lenses vary in scale |

In short, **measurements on an uncalibrated image cannot be trusted**. "I measured it in pixels"
guarantees nothing in millimetres.
#### English: When one scale factor is valid

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：輪郭の面積を mm² で表す
**日本語：** Part 3 で見つけた輪郭の面積を、`mm²` に変換して表示してください。

ヒント：面積の変換は `(px_per_mm ** 2)` で割ります（長さは 1 乗、面積は 2 乗）。

**English:** Convert the contour areas from Part 3 into `mm²`.
Hint: divide by `px_per_mm ** 2` (length scales linearly, area quadratically).
#### English: Exercise 1 (basic): express contour area in mm²
Done when each contour is printed with both px² and mm².

In [ ]:
# 演習 1 / Exercise 1
if "kept" in dir() and calibration_image is not None:
    for index, contour in enumerate(kept, start=1):
        area_px = cv2.contourArea(contour)
        area_mm2 = 0.0     # <- 計算してください / compute
        print(f"{index}: {area_px:8.1f} px2  ->  {area_mm2:.4f} mm2")

#### 日本語：演習 2（中級）：しきい値を変えて輪郭の数を比べる
**日本語：** しきい値を 100 / 150 / 180 / 200 と変えて、
**見つかった輪郭の数** と **最大の輪郭の面積** を表にまとめてください。
どの値が一番「欠陥だけを捉えて」いますか？理由も書いてください。

**English:** For thresholds 100 / 150 / 180 / 200, record the **number of contours** and the
**largest contour area**. Which value captures only the defects? Explain why.
#### English: Exercise 2 (intermediate): compare contour counts for several thresholds
Done when the table has four rows and you justify one chosen threshold.

In [ ]:
# 演習 2 / Exercise 2
if image_bgr is not None:
    for value in [100, 150, 180, 200]:
        _, test_binary = cv2.threshold(gray, value, 255, cv2.THRESH_BINARY_INV)
        test_contours, _ = cv2.findContours(test_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        areas = [cv2.contourArea(c) for c in test_contours if cv2.contourArea(c) >= 5]
        largest = max(areas) if areas else 0.0
        print(f"threshold={value:>3}  contours={len(areas):>3}  largest={largest:8.1f} px2")

#### 日本語：演習 3（発展）：測定誤差を考察する
**日本語：** 0.3 mm の欠陥を 1 px/mm で測るとします。

1. 0.3 mm は何ピクセルになりますか。
2. ピクセルは整数なので、0.3 px をどう扱うことになりますか。
3. 0.3 mm を 3 画素で捉えるには、何 px/mm 以上が必要ですか。
4. 誤差を小さくする方法を 3 つ挙げてください（カメラ・照明・校正の観点で）。

**English:** Suppose you image a 0.3 mm defect at 1 px/mm.

1. How many pixels does 0.3 mm occupy?
2. Pixels are integers — what happens to 0.3 px?
3. How many px/mm are needed for the defect to cover 3 pixels?
4. List three ways to reduce the error (camera, lighting, calibration).
#### English: Exercise 3 (challenge): discuss measurement error
Done when all four answers are written and question 3 states a number.

In [ ]:
# 演習 3 / Exercise 3
defect_mm = 0.3
current_px_per_mm = 1.0

# 1. 0.3 mm は何ピクセルか / how many pixels
pixels_at_current_scale = 0.0        # <- 計算 / compute
# 2. 整数化するとどうなるか / what happens when rounded to whole pixels
# 3. 3 画素で捉えるのに必要な px/mm / px per mm needed for 3 pixels
required_px_per_mm = 0.0             # <- 計算 / compute
# 4. 誤差を小さくする方法 / ways to reduce the error
#    - 
#    - 
#    - 

print("1:", pixels_at_current_scale)
print("2:", "")
print("3:", required_px_per_mm, "px/mm")
print("4:", "")

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. OpenCV の画像と Matplotlib の画像で、色の順番はどう違いますか。変換を忘れると何が起きますか。
2. `cv2.imread` が存在しないファイルを読んだとき、エラーになりますか。
3. `THRESH_BINARY` と `THRESH_BINARY_INV` の違いは何ですか。なぜ後者を使いましたか。
4. しきい値を上げすぎると、誤検出と見逃しのどちらが増えますか。
5. 「校正していない画像の測定値は信用できない」のはなぜですか。カメラの距離を例に説明してください。

**English:**

1. How do colour orders differ between OpenCV and Matplotlib, and what happens if you forget to convert?
2. Does `cv2.imread` raise an error for a missing file?
3. What is the difference between `THRESH_BINARY` and `THRESH_BINARY_INV`, and why did we use the latter?
4. If you raise the threshold too much, do false alarms or misses increase?
5. Why can measurements from an uncalibrated image not be trusted? Use camera distance as an example.
#### English: Understanding questions
Model answers: `docs/instructor/day02_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `image_bgr` が `None` になる | `image_bgr` is `None` | パスが違うか、素材が未生成です。`ASSETS / "defect_samples" / "defect_multi.png"` を確認してください。 | Wrong path, or the assets were never generated. Check the path. |
| `cv2.imwrite` が `False` を返す | `cv2.imwrite` returns `False` | 保存先フォルダが存在しません。`mkdir(parents=True, exist_ok=True)` を先に実行してください。 | The target folder does not exist. Run `mkdir(parents=True, exist_ok=True)` first. |
| 色がおかしい（赤と青が入れ替わる） | Colours look swapped (red/blue) | `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` を忘れています。 | You forgot `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)`. |
| `findContours` が 3 つの値を返すと言われる | `findContours` allegedly returns three values | OpenCV 3 系の情報です。OpenCV 4 系では 2 つ（contours, hierarchy）です。 | That is OpenCV 3 documentation. In OpenCV 4 it returns two values. |
| 輪郭が多すぎる／ノイズだらけ | Too many contours / mostly noise | `cv2.contourArea` で小さい輪郭を捨ててください（例：5 px² 未満）。 | Filter small contours with `cv2.contourArea` (e.g. drop below 5 px²). |
| `ModuleNotFoundError: No module named 'cv2'` | `ModuleNotFoundError: No module named 'cv2'` | カーネルが違うか、`opencv-python` が未導入です。`verify_environment.py` で確認してください。 | Wrong kernel, or `opencv-python` is missing. Run `verify_environment.py`. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] 画像を読み込み、保存先に保存できた
      - [ ] Read an image and saved a copy
- [ ] グレースケールと 2 値化の結果を目で確認した
      - [ ] Inspected the greyscale and binary results
- [ ] 輪郭を検出し、面積と外接四角を表示した
      - [ ] Detected contours and printed area and box
- [ ] 2 点間の距離を px で計算できた
      - [ ] Computed a distance in pixels
- [ ] px/mm を計算し、mm に換算できた
      - [ ] Computed px/mm and converted to mm
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [OpenCV 公式チュートリアル](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
- [cv2.threshold](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)
- [cv2.findContours](https://docs.opencv.org/4.x/d3/d05/tutorial_py_table_of_contents_contours.html)
- [カメラ校正の考え方](https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `cv2.adaptiveThreshold` を使ってみてください。照明のムラがある画像で、固定しきい値より良い結果になる理由を考えてください。

**English:** Try `cv2.adaptiveThreshold` and explain why it can beat a fixed threshold when the lighting is uneven.